In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    when,
    collect_list,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_real_estate_building_qca(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "RealEstateBuildingId String, SourceSystem String, IngestionTimestamp timestamp, BatchId String, RealEstateBuildingKey string, RealEstateBuildingType string,"
        #     "RealEstateKey string, RealEstateCepalKey String, RealEstateStatusCode String, RealEstateStatus String, SourceTableName String, RealEstateComments String, RealEstateActive String,"
        #     "RealEstateDescription String, RealEstateSubCategoryCode String, RealEstateSubCategory String, RealEstateCategoryCode String, RealEstateCategory String,"
        #     "RealEstateAddress1 String, RealEstateAddress2 String, RealEstateBuildingNumber String, RealEstateEntranceAddress String, RealEstatePostalCode String,"
        #     "RealEstateGeolocation String, RealEstateLocationLambda double, RealEstateLocationPhi double, RealEstateEncumbrances String, RealEstateEncumbrancesStatus String, RealEstateEncumbrancesDetails String, "
        #     "RealEstateLandArea double, RealEstateContractArea double, RealEstateNetArea double, RealEstateGlaArea double,  RealEstateOccupancyTypeCode string, RealEstateOccupancyType string, "
        #     "RealEstateOccupancyRate double, RealEstateSurfaceArea double, RealEstatePlottedArea double,"
        #     "RealEstatePlottedDate date, RealEstateAcquisitionValue decimal(19,4), RealEstateBookValue decimal(19,4), RealEstateApprovedIASValue decimal(19,4), RealEstateFiscalValue decimal(19,4),"
        #     "RealEstateMarketValue decimal(19,4), RealEstateEvaluationDate date, RealEstateEvaluationTypeCode String, RealEstateEvaluationType String, RealEstateForcedSaleValue decimal(19,4),"
        #     "RealEstateSaleAmount decimal(19,4), RealEstateSaleVATAmount decimal(19,4), RealEstateSaleTotalAmount decimal(19,4), RealEstateSaleDate date, RealEstateDateEntered date,"
        #     "RealEstateFirstOfferPrice decimal(19,4), RealEstateMinimumPrice decimal(19,4), RealEstateMaximumPrice decimal(19,4), RealEstateAuctionAmount decimal(19,4), RealEstateConstructionPercentage String,"
        #     "RealEstateLegalIssuesFlag boolean, RealEstateValueDate date, RealEstateGroupMarketValueDate date, RealEstateValueTypeCode String, RealEstateValueType String, RealEstateGroupMarketValue decimal(19,2),"
        #     "RealEstateGroupMarketValueTypeCode String, RealEstateGroupMarketValueType String, RealEstateConstructionYear String, RealEstateLandRegistryCode String, RealEstateLandRegistryOffice String,"
        #     "RealEstateLandRegistry String, RealEstateUsageCodeTypeCode String, RealEstateUsageCodeType String, RealEstateSiteCode String, RealEstatePrimaryLandLocation String, RealEstateTotalTax decimal(19,2),"
        #     "RealEstateOperatingCost decimal(19,2), RealEstateArea String, RealEstateCity String, RealEstateIsEAPFlag boolean, RealEstateDistrict String, RealEstateDivision String,"
        #     "RealEstateMunicipality String, RealEstateCounty String, RealEstateMarketabilityCode String, RealEstateKaekCode String, RealEstateKaekOffice String, RealEstateHotelFlag boolean,"
        #     "RealEstateHotelStars String, RealEstateHotelRooms int, RealEstateSpecialConditions String, RealEstateAuctionDate timestamp, RealEstateAuctionCode String, RealEstatePlaceCode String,"
        #     "RealEstateConstructionStatusCode String, RealEstateConstructionStatus String, RealEstateBuildingName String, RealEstateBuildingActive String, RealEstateBuildingLandRegistryUnit String,"
        #     "RealEstateBuildingLandBookNumber String, RealEstateBuildingOldLandRegistryUnit String, RealEstatePropertyBuildingNumber String, RealEstateBuildingOldLandBookNumber String,"
        #     "RealEstateBuildingPreviousRegistryUnit String, RealEstateBuildingTopographicNumber String, RealEstateBuildingBuildingTypeCode String, RealEstateBuildingBuildingType String,"
        #     "RealEstateBuildingDescription String, RealEstateBuildingFloorRawValue double, RealEstateBuildingContractArea double, RealEstateBuildingNetArea double, RealEstateBuildingSurfaceArea2 double,"
        #     "RealEstateBuildingGrossRentableArea double, RealEstateBuildingOccupancyRate double, RealEstateBuildingRenovationYear int, RealEstateBuildingPermitNumber String, RealEstateLandOwnership string,"
        #     "RealEstateBuildingPermitDate timestamp, RealEstateBuildingFacadeArea double, RealEstateBuildingFreeHeight double, RealEstateBuildingTechnicalNotes String, RealEstateBuildingOtherAuxiliaryArea double,"
        #     "RealEstateBuildingParkingSpace int, RealEstateBuildingStorageArea double, RealEstateBuildingLoftArea double, RealEstateBuildingDetachmentArea String, RealEstateBuildingPowerNumber String,"
        #     "RealEstateBuildingDateEntered timestamp, RealEstateBuildingEnteredBy int, RealEstateBuildingAreaTitle double, RealEstateBuildingAreaLicense double, RealEstateBuildingAreaSettled double,"
        #     "RealEstateBuildingUsageCode String, RealEstateBuildingParkingArea double, RealEstateBuildingWarehouseNumber int, RealEstateIsEligibleFlag boolean, RealEstateCountry string,"
        #     "CreatedDate string, CreatedBy string, LastUpdatedDate string, LastUpdatedBy string,  LocationTrackingCode string, LocationTracking string, BusinessDate date, RealEstateBuildingFloorRawValueDescription string"

        # )
        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.RealEstateBuilding"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "qca",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== fsad_real_estate_investigations_properties ==================

        fsad_real_estate_investigations_properties_extra_filters = None

        if fsad_real_estate_investigations_properties_extra_filters is not None:
            fsad_real_estate_investigations_properties_extra_filters.append(date_filter)
            fsad_real_estate_investigations_properties_final_filters = " AND ".join(
                fsad_real_estate_investigations_properties_extra_filters
            )
        else:
            fsad_real_estate_investigations_properties_final_filters = date_filter

        fsad_real_estate_investigations_properties_selected_fields = [
            "reip_code",
            "rein_code",
            "reip_property_status_intcode",
            "reip_property_comments",
            "reip_prop_active_flag",
            "reip_category_intcode",
            "reip_address",
            "reip_postal_code",
            "reip_prefecture",
            "reip_rest_country",
            "reip_city",
            "reip_land_area",
            "reip_land_percentage",
            "reip_land_registry_num",
            "businessdate",
        ]

        fsad_real_estate_investigations_properties = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fsad_real_estate_investigations_properties",
            fsad_real_estate_investigations_properties_final_filters,
            fsad_real_estate_investigations_properties_selected_fields,
        )

        # ================== fs_real_estate_investigations ==================

        fs_real_estate_investigations_extra_filters = None

        if fs_real_estate_investigations_extra_filters is not None:
            fs_real_estate_investigations_extra_filters.append(date_filter)
            fs_real_estate_investigations_final_filters = " AND ".join(
                fs_real_estate_investigations_extra_filters
            )
        else:
            fs_real_estate_investigations_final_filters = date_filter

        fs_real_estate_investigations_selected_fields = [
            "rein_land_registry",
            "rein_code",
            "cust_code",
        ]

        fs_real_estate_investigations = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fs_real_estate_investigations",
            fs_real_estate_investigations_final_filters,
            fs_real_estate_investigations_selected_fields,
        )
        # ================== at_properties ==================

        at_properties_extra_filters = None

        if at_properties_extra_filters is not None:
            at_properties_extra_filters.append(date_filter)
            at_properties_final_filters = " AND ".join(
                at_properties_extra_filters
            )
        else:
            at_properties_final_filters = date_filter

        at_properties_selected_fields = [
            "PRO_COMMENTS",
            "PRO_ACTIVE_FLAG",
            "PRO_STATUS",
            "PRO_CODE",
            "PRO_CREATED",
            "PRO_CREATED_BY",
            "PRO_LAST_UPD",
            "PRO_LAST_UPD_BY",               
            "CUST_CODE",
            "REST_CODE",
        ]
            
        at_properties = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
           "qca_at_properties",
            at_properties_final_filters,
            at_properties_selected_fields,
        )

        at_properties = (
                at_properties
                .withColumnRenamed("REST_CODE", "REST_CODE_atp")
                .withColumnRenamed("CUST_CODE", "CUST_CODE_atp")
                .withColumnRenamed("PRO_CODE", "PRO_CODE_atp")
                        )

        # ========================== at_real_estates ==========================

        at_real_estates_owner_extra_filters = None

        if at_real_estates_owner_extra_filters is not None:
            at_real_estates_owner_extra_filters.append(date_filter)
            at_real_estates_final_filters = " AND ".join(
                at_real_estates_owner_extra_filters
            )
        else:
            at_real_estates_final_filters = date_filter

        at_real_estates_selected_fields = [
            "REST_CODE",
            "REST_EXTERNAL_COMMENTS",
            "REST_ACTIVE",
            "REST_OWNERSHIP_TYPE_INTCODE",
            "REST_CATEGORY_CODE",
            "REST_ADDRESS",
            "REST_POSTAL_CODE",
            "REST_AREA",
            "REST_LAND_AREA",
            "REST_INITIAL_BOOK_VALUE",
            "REST_INITIAL_MARKET_VALUE",
            "REST_VALUE_ESTIMATION_DATE",
            "REST_LAND_REGISTRY_NUMBER",
            "REST_TOPONYM",
            "REST_CITY",
            "REST_DISTRICT",
            "REST_MUNICIPALITY",
            "REST_FLOORS",
            "REST_COUNTRY",
            "REST_CREATED",
            "REST_CREATED_BY",
            "REST_LAST_UPD",
            "REST_LAST_UPD_BY",
            "REST_EXTERNAL_ID",
            "REST_PREFECTURE",
        ]

        at_real_estates = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_at_real_estates",
            at_real_estates_final_filters,
            at_real_estates_selected_fields,
        )

        at_real_estates = (
                at_real_estates
                .withColumnRenamed("REST_CODE", "REST_CODE_atre")
                .withColumnRenamed("REST_CATEGORY_CODE", "REST_CATEGORY_CODE_atre")
                .withColumnRenamed("REST_LAND_AREA", "REST_LAND_AREA_atre")
                .withColumnRenamed("REST_INITIAL_MARKET_VALUE", "REST_INITIAL_MARKET_VALUE_atre")
                .withColumnRenamed("REST_LAND_REGISTRY_NUMBER", "REST_LAND_REGISTRY_NUMBER_atre")
                .withColumnRenamed("REST_TOPONYM", "REST_TOPONYM_atre")
                .withColumnRenamed("REST_EXTERNAL_ID", "REST_EXTERNAL_ID_atre")
                .withColumnRenamed("REST_PREFECTURE", "REST_PREFECTURE_atre")
                        )
        
        # ================== qca_at_real_estate_floors ==================

        at_real_estates_floors_extra_filters = None

        if at_real_estates_floors_extra_filters is not None:
            at_real_estates_floors_extra_filters.append(date_filter)
            at_real_estates_floors_final_filters = " AND ".join(
                at_real_estates_floors_extra_filters
            )
        else:
            at_real_estates_floors_final_filters = date_filter

        at_real_estates_floors_selected_fields = [
            "REST_CODE",
            "REFL_FLOOR",
            "REFL_CODE",
        ]
 
        at_real_estates_floors = load_landed_delta_table(
            spark,
            catalog_landed,  
            "qualcoalpha_atrealestatefloors",
            at_real_estates_floors_final_filters,
            at_real_estates_floors_selected_fields,
        )

        at_real_estates_floors = (
            at_real_estates_floors.withColumnRenamed("REST_CODE", "REST_CODE_FLOORS")
        )

        # ================== fsad_properties_real_estates ==================

        fsad_properties_real_estates_extra_filters = None

        if fsad_properties_real_estates_extra_filters is not None:
            fsad_properties_real_estates_extra_filters.append(date_filter)
            fsad_properties_real_estates_final_filters = " AND ".join(
                fsad_properties_real_estates_extra_filters
            )
        else:
            fsad_properties_real_estates_final_filters = date_filter

        fsad_properties_real_estates_selected_fields = [
            "PRO_ASSET_TYPE_CODE",
            "REST_CATEGORY_CODE",
            "REST_LAND_AREA",
            "REST_CURRENT_BOOK_VALUE_LCY",
            "REST_INITIAL_MARKET_VALUE",
            "REVE_ESTIMATION_DATE",
            "REST_LAND_REGISTRY_NUMBER",
            "REST_PREFECTURE",
            "REST_TOPONYM",
            "REST_CODE",
            "REST_EXTERNAL_ID",
            "PRO_CODE",
            "businessdate",
        ]

        fsad_properties_real_estates = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fsad_properties_real_estates",
            fsad_properties_real_estates_final_filters,
            fsad_properties_real_estates_selected_fields,
        )

        # ================== real_estate_building_qca_fs ==================
        real_estate_building_qca_fs = at_real_estates.join(
            at_real_estates_floors,
            (
                at_real_estates["REST_CODE_atre"] == at_real_estates_floors["REST_CODE_FLOORS"]
            ),
            "left",
        ).join(
            at_properties,
            (
                at_real_estates["REST_CODE_atre"]
                == at_properties["REST_CODE_atp"]
            ),
            "left",
        ).join(
            fs_real_estate_investigations,
            (
                at_properties["CUST_CODE_atp"]
                == fs_real_estate_investigations["cust_code"]
            ),
            "left",
        ).join(
            fsad_real_estate_investigations_properties,
            (
                fs_real_estate_investigations["rein_code"]
                == fsad_real_estate_investigations_properties["rein_code"]
            ),
            "inner",
        )

        real_estate_building_qca_fs_primary_key_fields = [
            at_real_estates.REST_CODE_atre,
            at_properties.PRO_CODE_atp,
            at_properties.CUST_CODE_atp,
            at_real_estates_floors.REFL_CODE,
            lit("EAP RealEstate"),
            lit("qca"),
        ]

        real_estate_building_qca_fs = (
            real_estate_building_qca_fs.withColumn(
                "RealEstateBuildingId",
                sha2(
                    concat_ws(
                        ";",
                        *real_estate_building_qca_fs_primary_key_fields,
                    ),
                    256,
                ),
            )
            .withColumn(
                "RealEstateBuildingKey",
                concat_ws(
                    ";",
                    fsad_real_estate_investigations_properties.reip_code,
                    real_estate_building_qca_fs.REFL_CODE,
                    lit("EAP RealEstate"),
                ),
            )
            .withColumn(
                "RealEstateActive",
                when(
                    fsad_real_estate_investigations_properties.reip_prop_active_flag
                    == 1,
                    "Active",
                )
                .when(
                    fsad_real_estate_investigations_properties.reip_prop_active_flag
                    == 0,
                    "Inactive",
                )
                .otherwise("N/A"),
            )
            .withColumn(
                "RealEstateBuildingActive",
                when(
                    fsad_real_estate_investigations_properties.reip_prop_active_flag
                    == 1,
                    "Active",
                )
                .when(
                    fsad_real_estate_investigations_properties.reip_prop_active_flag
                    == 0,
                    "Inactive",
                )
                .otherwise("N/A"),
            )
        )


        # ================== Selection of Final Columns for FS ==================

        real_estate_building_qca_fs = real_estate_building_qca_fs.select(
            real_estate_building_qca_fs.RealEstateBuildingId.cast("String").alias("RealEstateBuildingId"),
            lit("qca").cast("String").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).cast("timestamp").alias("IngestionTimestamp"),
            lit(None).cast("String").alias("BatchId"),
            real_estate_building_qca_fs.RealEstateBuildingKey.cast("string").alias( "RealEstateBuildingKey"),
            lit("EAP Real Estate").alias("RealEstateBuildingType"),
            real_estate_building_qca_fs.REST_CODE_atre.cast("String").alias("RealEstateKey"), 
            real_estate_building_qca_fs.REST_EXTERNAL_ID_atre.cast("String").alias("RealEstateCepalKey"), 
            real_estate_building_qca_fs.reip_property_status_intcode.cast("String").alias("RealEstateStatusCode"),
            real_estate_building_qca_fs.reip_property_status_intcode.cast("String").alias("RealEstateStatus"),
            lit(None).cast("String").alias("SourceTableName"),
            real_estate_building_qca_fs.REST_EXTERNAL_COMMENTS.cast("String").alias("RealEstateComments"), 
            real_estate_building_qca_fs.REST_ACTIVE.cast("String").alias("RealEstateActive"),
            real_estate_building_qca_fs.REST_OWNERSHIP_TYPE_INTCODE.cast("String").alias("RealEstateDescription"), 
            lit(None).cast("String").alias("RealEstateSubCategoryCode"), 
            lit(None).cast("String").alias("RealEstateSubCategory"),
            real_estate_building_qca_fs.REST_CATEGORY_CODE_atre.cast("String").alias("RealEstateCategoryCode"), 
            real_estate_building_qca_fs.REST_CATEGORY_CODE_atre.cast("String").alias("RealEstateCategory"),
            lit(None).cast("string").alias("RealEstateInsuranceCategory"),
            real_estate_building_qca_fs.REST_ADDRESS.cast("String").alias("RealEstateAddress1"),               
            lit(None).alias("RealEstateAddress2"),
            lit(None).cast("String").alias("RealEstateBuildingNumber"),
            real_estate_building_qca_fs.reip_address.cast("String").alias( "RealEstateEntranceAddress"),
            real_estate_building_qca_fs.REST_POSTAL_CODE.cast("String").alias("RealEstatePostalCode"), 
            lit(None).cast("String").alias("RealEstateGeolocation"),
            lit(None).cast("double").alias("RealEstateLocationLambda"),
            lit(None).cast("double").alias("RealEstateLocationPhi"),
            lit(None).cast("String").alias("RealEstateEncumbrances"),
            lit(None).cast("String").alias("RealEstateEncumbrancesStatus"),
            lit(None).cast("String").alias("RealEstateEncumbrancesDetails"),
            real_estate_building_qca_fs.REST_LAND_AREA_atre.cast("double").alias("RealEstateLandArea"),
            lit(None).cast("double").alias("RealEstateContractArea"),
            lit(None).alias("RealEstateNetArea"),
            lit(None).cast("double").alias("RealEstateGlaArea"),
            lit(None).cast("string").alias("RealEstateOccupancyTypeCode"),
            lit(None).cast("string").alias("RealEstateOccupancyType"),
            real_estate_building_qca_fs.reip_land_percentage.cast("double").alias("RealEstateOccupancyRate"),
            lit(None).cast("double").alias("RealEstateSurfaceArea"),
            lit(None).alias("RealEstatePlottedArea"),
            lit(None).cast("date").alias("RealEstatePlottedDate"),
            lit(None).cast("decimal(19,4)").alias("RealEstateAcquisitionValue"),
            real_estate_building_qca_fs.REST_INITIAL_BOOK_VALUE.alias("RealEstateBookValue"), 
            lit(None).cast("decimal(19,4)").alias("RealEstateApprovedIASValue"),
            lit(None).alias("RealEstateFiscalValue"),
            real_estate_building_qca_fs.REST_INITIAL_MARKET_VALUE_atre.alias("RealEstateMarketValue"), 
            real_estate_building_qca_fs.REST_VALUE_ESTIMATION_DATE.cast("date").alias("RealEstateEvaluationDate"), 
            lit(None).cast("String").alias("RealEstateEvaluationTypeCode"),
            lit(None).cast("String").alias("RealEstateEvaluationType"),
            lit(None).cast("decimal(19,4)").alias("RealEstateForcedSaleValue"),
            lit(None).cast("decimal(19,4)").alias("RealEstateSaleAmount"),
            lit(None).cast("decimal(19,4)").alias("RealEstateSaleVATAmount"),
            lit(None).cast("decimal(19,4)").alias("RealEstateSaleTotalAmount"),
            lit(None).cast("date").alias("RealEstateSaleDate"),
            lit(None).cast("date").alias("RealEstateDateEntered"),
            lit(None).cast("decimal(19,4)").alias("RealEstateFirstOfferPrice"),
            lit(None).cast("decimal(19,4)").alias("RealEstateMinimumPrice"),
            lit(None).cast("decimal(19,4)").alias("RealEstateMaximumPrice"),
            lit(None).cast("decimal(19,4)").alias("RealEstateAuctionAmount"),
            lit(None).cast("String").alias("RealEstateConstructionPercentage"),
            lit(None).cast("boolean").alias("RealEstateLegalIssuesFlag"),
            lit(None).cast("date").alias("RealEstateValueDate"), 
            lit(None).cast("date").alias("RealEstateGroupMarketValueDate"),
            lit(None).cast("String").alias("RealEstateValueTypeCode"),
            lit(None).cast("String").alias("RealEstateValueType"),
            lit(None).cast("decimal(19,2)").alias("RealEstateGroupMarketValue"),
            lit(None).cast("String").alias("RealEstateGroupMarketValueTypeCode"),
            lit(None).cast("String").alias("RealEstateGroupMarketValueType"),
            lit(None).cast("String").alias("RealEstateConstructionYear"),
            real_estate_building_qca_fs.REST_LAND_REGISTRY_NUMBER_atre.cast("String").alias("RealEstateLandRegistryCode"),  
            lit(None).cast("String").alias("RealEstateLandRegistryOffice"), 
            real_estate_building_qca_fs.rein_land_registry.cast("String").alias("RealEstateLandRegistry"), 
            lit(None).cast("String").alias("RealEstateUsageCodeTypeCode"),
            lit(None).cast("String").alias("RealEstateUsageCodeType"),
            lit(None).cast("String").alias("RealEstateSiteCode"),
            lit(None).cast("String").alias("RealEstatePrimaryLandLocation"),
            lit(None).cast("decimal(19,2)").alias("RealEstateTotalTax"),
            lit(None).cast("decimal(19,2)").alias("RealEstateOperatingCost"),
            real_estate_building_qca_fs.REST_PREFECTURE_atre.cast("String").alias("RealEstateArea"),     
            real_estate_building_qca_fs.REST_TOPONYM_atre.cast("String").alias("RealEstateCity"),             
            real_estate_building_qca_fs.reip_city.cast("boolean").alias("RealEstateIsEAPFlag"),
            real_estate_building_qca_fs.REST_DISTRICT.cast("String").alias("RealEstateDistrict"),
            lit(None).cast("String").alias("RealEstateDivision"),
            real_estate_building_qca_fs.REST_MUNICIPALITY.cast("String").alias("RealEstateMunicipality"), 
            lit(None).cast("String").alias("RealEstateCounty"),
            lit(None).cast("String").alias("RealEstateMarketabilityCode"),
            lit(None).cast("String").alias("RealEstateKaekCode"),
            lit(None).cast("String").alias("RealEstateKaekOffice"),
            lit(None).cast("boolean").alias("RealEstateHotelFlag"),
            lit(None).cast("String").alias("RealEstateHotelStars"),
            lit(None).cast("int").alias("RealEstateHotelRooms"),
            lit(None).cast("String").alias("RealEstateSpecialConditions"),
            lit(None).cast("timestamp").alias("RealEstateAuctionDate"),
            lit(None).cast("String").alias("RealEstateAuctionCode"),
            lit(None).cast("String").alias("RealEstatePlaceCode"),
            lit(None).cast("String").alias("RealEstateConstructionStatusCode"),
            lit(None).cast("String").alias("RealEstateConstructionStatus"),
            lit(None).cast("String").alias("RealEstateBuildingName"),
            real_estate_building_qca_fs.RealEstateBuildingActive.cast("String").alias("RealEstateBuildingActive"),
            lit(None).cast("String").alias("RealEstateBuildingLandRegistryUnit"),
            lit(None).cast("String").alias("RealEstateBuildingLandBookNumber"),
            lit(None).cast("String").alias("RealEstateBuildingOldLandRegistryUnit"),
            lit(None).cast("String").alias("RealEstatePropertyBuildingNumber"),
            lit(None).cast("String").alias("RealEstateBuildingOldLandBookNumber"),
            lit(None).cast("String").alias("RealEstateBuildingPreviousRegistryUnit"),
            lit(None).cast("String").alias("RealEstateBuildingTopographicNumber"),
            lit(None).cast("String").alias("RealEstateBuildingBuildingTypeCode"),
            lit(None).cast("String").alias("RealEstateBuildingBuildingType"),
            real_estate_building_qca_fs.REST_OWNERSHIP_TYPE_INTCODE.cast("String").alias("RealEstateBuildingDescription"), 
            lit(None).cast("double").alias("RealEstateBuildingFloorRawValue"),
            lit(None).cast("double").alias("RealEstateBuildingContractArea"),
            real_estate_building_qca_fs.REST_AREA.cast("double").alias("RealEstateBuildingNetArea"),
            lit(None).cast("double").alias("RealEstateBuildingSurfaceArea2"),
            lit(None).cast("double").alias("RealEstateBuildingGrossRentableArea"),
            lit(None).cast("double").alias("RealEstateBuildingOccupancyRate"),
            lit(None).cast("int").alias("RealEstateBuildingRenovationYear"),
            lit(None).cast("String").alias("RealEstateBuildingPermitNumber"),
            lit(None).cast("timestamp").alias("RealEstateBuildingPermitDate"),
            lit(None).cast("double").alias("RealEstateBuildingFacadeArea"),
            lit(None).cast("double").alias("RealEstateBuildingFreeHeight"),
            lit(None).cast("String").alias("RealEstateBuildingTechnicalNotes"),
            lit(None).cast("double").alias("RealEstateBuildingOtherAuxiliaryArea"),
            lit(None).cast("int").alias("RealEstateBuildingParkingSpace"),
            lit(None).cast("double").alias("RealEstateBuildingStorageArea"),
            lit(None).cast("double").alias("RealEstateBuildingLoftArea"),
            lit(None).cast("String").alias("RealEstateBuildingDetachmentArea"),
            lit(None).cast("String").alias("RealEstateBuildingPowerNumber"),
            lit(None).cast("timestamp").alias("RealEstateBuildingDateEntered"),
            lit(None).cast("int").alias("RealEstateBuildingEnteredBy"),
            lit(None).cast("double").alias("RealEstateBuildingAreaTitle"),
            lit(None).cast("double").alias("RealEstateBuildingAreaLicense"),
            lit(None).cast("double").alias("RealEstateBuildingAreaSettled"),
            lit(None).cast("String").alias("RealEstateBuildingUsageCode"),
            lit(None).cast("double").alias("RealEstateBuildingParkingArea"),
            lit(None).cast("int").alias("RealEstateBuildingWarehouseNumber"),
            lit(None).cast("boolean").alias("RealEstateIsEligibleFlag"),
            real_estate_building_qca_fs.REST_COUNTRY.cast("string").alias("RealEstateCountry"), 
            real_estate_building_qca_fs.REST_CREATED.cast("string").alias("CreatedDate"), 
            real_estate_building_qca_fs.REST_CREATED_BY.cast("string").alias("CreatedBy"), 
            real_estate_building_qca_fs.REST_LAST_UPD.cast("string").alias("LastUpdatedDate"), 
            real_estate_building_qca_fs.REST_LAST_UPD_BY.cast("string").alias("LastUpdatedBy"), 
            lit(None).cast("string").alias("LocationTrackingCode"),
            lit(None).cast("string").alias("LocationTracking"),
            real_estate_building_qca_fs.businessdate.cast("date").alias("BusinessDate"),
            real_estate_building_qca_fs.REFL_FLOOR.cast("string").alias("RealEstateBuildingFloorRawValueDescription"),

        )

        # ================== real_estate_building_qca_fsad ==================

        real_estate_building_qca_fsad = at_real_estates.join(
            at_real_estates_floors,
            (
                at_real_estates["REST_CODE_atre"] == at_real_estates_floors["REST_CODE_FLOORS"]
            ),
            "left",
        ).join(
            at_properties,
            (at_real_estates["REST_CODE_atre"] == at_properties["REST_CODE_atp"]),
            "left",
        ).join(
            fsad_properties_real_estates,
            (at_properties["PRO_CODE_atp"]== fsad_properties_real_estates["pro_code"]),
            "left",
        )

        real_estate_building_qca_fsad_primary_key_fields = [
            at_real_estates.REST_CODE_atre,
            at_real_estates_floors.REFL_CODE,
            lit("Real Estate"),
            lit("qca"),
        ]

        real_estate_building_qca_fsad = (
            real_estate_building_qca_fsad.withColumn(
                "RealEstateBuildingId",
                sha2(
                    concat_ws(
                        ";",
                        *real_estate_building_qca_fsad_primary_key_fields,
                    ),
                    256,
                ),
            ).withColumn(
                "RealEstateBuildingKey",
                concat_ws(
                    ";",
                    fsad_properties_real_estates.REST_CODE,
                    real_estate_building_qca_fsad.REFL_CODE,
                    lit("Real Estate"),
                ),
            )
        )


        # ================== Selection of Final Columns for FSAD ==================

        real_estate_building_qca_fsad = real_estate_building_qca_fsad.select(
            real_estate_building_qca_fsad.RealEstateBuildingId.cast("String").alias("RealEstateBuildingId"),
            lit("qca").cast("String").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).cast("timestamp").alias("IngestionTimestamp"),
            lit(None).cast("String").alias("BatchId"),
            real_estate_building_qca_fsad.RealEstateBuildingKey.cast("string").alias("RealEstateBuildingKey"),
            lit("Real Estate").alias("RealEstateBuildingType"),
            real_estate_building_qca_fsad.REST_CODE_atre.cast("String").alias("RealEstateKey"), 
            real_estate_building_qca_fsad.REST_EXTERNAL_ID_atre.cast("String").alias("RealEstateCepalKey"),
            lit(None).cast("String").alias("RealEstateStatusCode"),  #real_estate_building_qca_fsad.PRO_ACTIVE_FLAG
            lit(None).cast("String").alias("RealEstateStatus"),  # real_estate_building_qca_fsad.RealEstateStatus
            lit(None).cast("String").alias("SourceTableName"),
            real_estate_building_qca_fsad.REST_EXTERNAL_COMMENTS.cast("String").alias("RealEstateComments"),  # real_estate_building_qca_fsad.PRO_COMMENTS
            #real_estate_building_qca_fsad.RealEstateActive.cast("String").alias(  # real_estate_building_qca_fsad.RealEstateActive
            #"RealEstateActive"),
            real_estate_building_qca_fsad.REST_ACTIVE.cast("String").alias("RealEstateActive"),
            real_estate_building_qca_fsad.REST_OWNERSHIP_TYPE_INTCODE.cast("String").alias("RealEstateDescription"),  # real_estate_building_qca_fsad.PRO_STATUS
            lit(None).cast("String").alias("RealEstateSubCategoryCode"), #real_estate_building_qca_fsad.PRO_ASSET_TYPE_CODE
            lit(None).cast("String").alias("RealEstateSubCategory"), #real_estate_building_qca_fsad.PRO_ASSET_TYPE_CODE            
            real_estate_building_qca_fsad.REST_CATEGORY_CODE_atre.cast("String").alias("RealEstateCategoryCode"), 
            real_estate_building_qca_fsad.REST_CATEGORY_CODE_atre.cast("String").alias("RealEstateCategory"),  
            lit(None).cast("string").alias("RealEstateInsuranceCategory"),
            real_estate_building_qca_fsad.REST_ADDRESS.cast("String").alias("RealEstateAddress1"), 
            lit(None).alias("RealEstateAddress2"),
            lit(None).cast("String").alias("RealEstateBuildingNumber"),
            lit(None).cast("String").alias("RealEstateEntranceAddress"),
            real_estate_building_qca_fsad.REST_POSTAL_CODE.cast("String").alias("RealEstatePostalCode"), 
            lit(None).cast("String").alias("RealEstateGeolocation"),
            lit(None).cast("double").alias("RealEstateLocationLambda"),
            lit(None).cast("double").alias("RealEstateLocationPhi"),
            lit(None).cast("String").alias("RealEstateEncumbrances"),
            lit(None).cast("String").alias("RealEstateEncumbrancesStatus"),
            lit(None).cast("String").alias("RealEstateEncumbrancesDetails"),
            real_estate_building_qca_fsad.REST_LAND_AREA_atre.cast("double").alias("RealEstateLandArea"),
            lit(None).cast("double").alias("RealEstateContractArea"),
            lit(None).alias("RealEstateNetArea"),
            lit(None).cast("double").alias("RealEstateGlaArea"),
            lit(None).cast("string").alias("RealEstateOccupancyTypeCode"),
            lit(None).cast("string").alias("RealEstateOccupancyType"),
            lit(None).alias("RealEstateOccupancyRate"),
            lit(None).cast("double").alias("RealEstateSurfaceArea"),
            lit(None).alias("RealEstatePlottedArea"),
            lit(None).cast("date").alias("RealEstatePlottedDate"),
            lit(None).cast("decimal(19,4)").alias("RealEstateAcquisitionValue"),
            real_estate_building_qca_fsad.REST_CURRENT_BOOK_VALUE_LCY.alias("RealEstateBookValue"),
            lit(None).cast("decimal(19,4)").alias("RealEstateApprovedIASValue"),
            lit(None).alias("RealEstateFiscalValue"),
            real_estate_building_qca_fsad.REST_INITIAL_MARKET_VALUE_atre.alias("RealEstateMarketValue"), 
            real_estate_building_qca_fsad.REVE_ESTIMATION_DATE.cast("date").alias("RealEstateEvaluationDate"), 
            lit(None).cast("String").alias("RealEstateEvaluationTypeCode"),
            lit(None).cast("String").alias("RealEstateEvaluationType"),
            lit(None).cast("decimal(19,4)").alias("RealEstateForcedSaleValue"),
            lit(None).cast("decimal(19,4)").alias("RealEstateSaleAmount"),
            lit(None).cast("decimal(19,4)").alias("RealEstateSaleVATAmount"),
            lit(None).cast("decimal(19,4)").alias("RealEstateSaleTotalAmount"),
            lit(None).cast("date").alias("RealEstateSaleDate"),
            lit(None).cast("date").alias("RealEstateDateEntered"),
            lit(None).cast("decimal(19,4)").alias("RealEstateFirstOfferPrice"),
            lit(None).cast("decimal(19,4)").alias("RealEstateMinimumPrice"),
            lit(None).cast("decimal(19,4)").alias("RealEstateMaximumPrice"),
            lit(None).cast("decimal(19,4)").alias("RealEstateAuctionAmount"),
            lit(None).cast("String").alias("RealEstateConstructionPercentage"),
            lit(None).cast("boolean").alias("RealEstateLegalIssuesFlag"),
            lit(None).cast("date").alias("RealEstateValueDate"), 
            lit(None).cast("date").alias("RealEstateGroupMarketValueDate"),
            lit(None).cast("String").alias("RealEstateValueTypeCode"),
            lit(None).cast("String").alias("RealEstateValueType"),
            lit(None).cast("decimal(19,2)").alias("RealEstateGroupMarketValue"),
            lit(None).cast("String").alias("RealEstateGroupMarketValueTypeCode"),
            lit(None).cast("String").alias("RealEstateGroupMarketValueType"),
            lit(None).cast("String").alias("RealEstateConstructionYear"),
            real_estate_building_qca_fsad.REST_LAND_REGISTRY_NUMBER_atre.cast("String").alias("RealEstateLandRegistryCode"), 
            lit(None).cast("String").alias("RealEstateLandRegistryOffice"), 
            lit(None).cast("String").alias("RealEstateLandRegistry"), 
            lit(None).cast("String").alias("RealEstateUsageCodeTypeCode"),
            lit(None).cast("String").alias("RealEstateUsageCodeType"),
            lit(None).cast("String").alias("RealEstateSiteCode"),
            lit(None).cast("String").alias("RealEstatePrimaryLandLocation"),
            lit(None).cast("decimal(19,2)").alias("RealEstateTotalTax"),
            lit(None).cast("decimal(19,2)").alias("RealEstateOperatingCost"),
            real_estate_building_qca_fsad.REST_PREFECTURE_atre.cast("String").alias("RealEstateArea"), 
            real_estate_building_qca_fsad.REST_TOPONYM_atre.cast("String").alias("RealEstateCity"), 
            lit(None).cast("boolean").alias("RealEstateIsEAPFlag"),
            real_estate_building_qca_fsad.REST_DISTRICT.cast("String").alias("RealEstateDistrict"), 
            lit(None).cast("String").alias("RealEstateDivision"),
            real_estate_building_qca_fsad.REST_MUNICIPALITY.cast("String").alias("RealEstateMunicipality"), 
            lit(None).cast("String").alias("RealEstateCounty"),
            lit(None).cast("String").alias("RealEstateMarketabilityCode"),
            lit(None).cast("String").alias("RealEstateKaekCode"),
            lit(None).cast("String").alias("RealEstateKaekOffice"),
            lit(None).cast("boolean").alias("RealEstateHotelFlag"),
            lit(None).cast("String").alias("RealEstateHotelStars"),
            lit(None).cast("int").alias("RealEstateHotelRooms"),
            lit(None).cast("String").alias("RealEstateSpecialConditions"),
            lit(None).cast("timestamp").alias("RealEstateAuctionDate"),
            lit(None).cast("String").alias("RealEstateAuctionCode"),
            lit(None).cast("String").alias("RealEstatePlaceCode"),
            lit(None).cast("String").alias("RealEstateConstructionStatusCode"),
            lit(None).cast("String").alias("RealEstateConstructionStatus"),
            lit(None).cast("String").alias("RealEstateBuildingName"),
            lit(None).cast("String").alias("RealEstateBuildingActive"),  # real_estate_building_qca_fsad.RealEstateBuildingActive
            lit(None).cast("String").alias("RealEstateBuildingLandRegistryUnit"),
            lit(None).cast("String").alias("RealEstateBuildingLandBookNumber"),
            lit(None).cast("String").alias("RealEstateBuildingOldLandRegistryUnit"),
            lit(None).cast("String").alias("RealEstatePropertyBuildingNumber"),
            lit(None).cast("String").alias("RealEstateBuildingOldLandBookNumber"),
            lit(None).cast("String").alias("RealEstateBuildingPreviousRegistryUnit"),
            lit(None).cast("String").alias("RealEstateBuildingTopographicNumber"),
            lit(None).cast("String").alias("RealEstateBuildingBuildingTypeCode"),
            lit(None).cast("String").alias("RealEstateBuildingBuildingType"),
            real_estate_building_qca_fsad.REST_OWNERSHIP_TYPE_INTCODE.cast("String").alias("RealEstateBuildingDescription"), 
            lit(None).cast("double").alias("RealEstateBuildingFloorRawValue"),
            lit(None).cast("double").alias("RealEstateBuildingContractArea"),
            real_estate_building_qca_fsad.REST_AREA.cast("double").alias("RealEstateBuildingNetArea"),
            lit(None).cast("double").alias("RealEstateBuildingSurfaceArea2"),
            lit(None).cast("double").alias("RealEstateBuildingGrossRentableArea"),
            lit(None).cast("double").alias("RealEstateBuildingOccupancyRate"),
            lit(None).cast("int").alias("RealEstateBuildingRenovationYear"),
            lit(None).cast("String").alias("RealEstateBuildingPermitNumber"),
            lit(None).cast("timestamp").alias("RealEstateBuildingPermitDate"),
            lit(None).cast("double").alias("RealEstateBuildingFacadeArea"),
            lit(None).cast("double").alias("RealEstateBuildingFreeHeight"),
            lit(None).cast("String").alias("RealEstateBuildingTechnicalNotes"),
            lit(None).cast("double").alias("RealEstateBuildingOtherAuxiliaryArea"),
            lit(None).cast("int").alias("RealEstateBuildingParkingSpace"),
            lit(None).cast("double").alias("RealEstateBuildingStorageArea"),
            lit(None).cast("double").alias("RealEstateBuildingLoftArea"),
            lit(None).cast("String").alias("RealEstateBuildingDetachmentArea"),
            lit(None).cast("String").alias("RealEstateBuildingPowerNumber"),
            lit(None).cast("timestamp").alias("RealEstateBuildingDateEntered"),
            lit(None).cast("int").alias("RealEstateBuildingEnteredBy"),
            lit(None).cast("double").alias("RealEstateBuildingAreaTitle"),
            lit(None).cast("double").alias("RealEstateBuildingAreaLicense"),
            lit(None).cast("double").alias("RealEstateBuildingAreaSettled"),
            lit(None).cast("String").alias("RealEstateBuildingUsageCode"),
            lit(None).cast("double").alias("RealEstateBuildingParkingArea"),
            lit(None).cast("int").alias("RealEstateBuildingWarehouseNumber"),
            lit(None).cast("boolean").alias("RealEstateIsEligibleFlag"),
            real_estate_building_qca_fsad.REST_COUNTRY.cast("string").alias("RealEstateCountry"), 
            real_estate_building_qca_fsad.REST_CREATED.cast("string").alias("CreatedDate"),  # real_estate_building_qca_fsad.PRO_CREATED
            real_estate_building_qca_fsad.REST_CREATED_BY.cast("string").alias("CreatedBy"),  # real_estate_building_qca_fsad.PRO_CREATED_BY
            real_estate_building_qca_fsad.REST_LAST_UPD.cast("string").alias("LastUpdatedDate"),  # real_estate_building_qca_fsad.PRO_LAST_UPD
            real_estate_building_qca_fsad.REST_LAST_UPD_BY.cast("string").alias("LastUpdatedBy"),  # real_estate_building_qca_fsad.PRO_LAST_UPD_BY
            lit(None).cast("string").alias("LocationTrackingCode"),
            lit(None).cast("string").alias("LocationTracking"),
            real_estate_building_qca_fsad.businessdate.cast("date").alias("BusinessDate"),
            real_estate_building_qca_fsad.REFL_FLOOR.cast("string").alias("RealEstateBuildingFloorRawValueDescription"),

        ).distinct()  # dimentioanility of source table is propertyownership so it is need to use distinct

        # ================== Append ==================
        real_estate_building_qca = real_estate_building_qca_fs.union(
            real_estate_building_qca_fsad
        )
        real_estate_building_qca.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_real_estate_building_qca(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)